# Tanzania Food Demand and Nutritional Adequacy

This notebook estimates a demand system and discusses the landscape of the nutritional diet and its adequacy using household food expenditure data from Tanzania (2019–20 and 2020–21 survey waves).

In [17]:
# Cell 1 — Environment setup (run once per session)
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'CFEDemands', '--quiet'])

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from cfe import Regression
import cfe

### Data [A]

The data is stored in a local Excel file (`Tanzania.xlsx`) with the following sheets:

- **Household Characteristics** — one row per household-wave, with demographic columns (household members by age/sex group and log household size).
- **Food Expenditures (2019-20)** and **Food Expenditures (2020-21)** — long-format records of household food expenditures, one row per household–food pair.

All sheets share the index columns `i` (household ID), `t` (survey wave), and `m` (market/region).

### Load raw sheets

In [19]:
DATA_FILE = 'Tanzania.xlsx'

# Household characteristics (filter to the two survey waves that have expenditure data)
d_raw = pd.read_excel(DATA_FILE, sheet_name='Household Characteristics')

# Food expenditures — two survey waves loaded separately and concatenated
x_raw_2019 = pd.read_excel(DATA_FILE, sheet_name='Food Expenditures (2019-20)')
x_raw_2020 = pd.read_excel(DATA_FILE, sheet_name='Food Expenditures (2020-21)')
x_raw = pd.concat([x_raw_2019, x_raw_2020], ignore_index=True)

print(f"Household characteristics rows : {len(d_raw):,}")
print(f"Food expenditure records       : {len(x_raw):,}")
print(f"Unique foods                   : {x_raw['j'].nunique()}")
print(f"Survey waves                   : {sorted(x_raw['t'].unique())}")

### Household characteristics dataframe (d)
Missing values in demographic columns are treated as zero (no members of that age/sex group).

In [20]:
# Keep only waves that have matching expenditure data
WAVES = ['2019-20', '2020-21']
d = d_raw[d_raw['t'].isin(WAVES)].copy()

# Consistency string dtypes across columns
for col in ['i', 't', 'm']:
    d[col] = d[col].astype(str)

d.columns.name = 'k'
d.set_index(['i', 't', 'm'], inplace=True)
d = d.fillna(0) 

print(f"d shape: {d.shape}")
d.head()

### Log-expenditure dataframe (y)

In [21]:
# Consistent string dtypes
for col in ['i', 't', 'm']:
    x_raw[col] = x_raw[col].astype(str)

# One column per food item
x = x_raw.pivot_table(
    index=['i', 't', 'm'],
    columns='j',
    values='Expenditure',
    aggfunc='sum'
)
x = x.replace(0, np.nan)  # Replace zeros with missing before taking logs
y = np.log(x)

print(f"y shape (households × foods): {y.shape}")
y.head()

### Re-format for regression

In [22]:
y = y.stack().dropna()
d = d.stack().dropna()

# Verify index structure
assert y.index.names == ['i', 't', 'm', 'j'], f"Unexpected y index: {y.index.names}"
assert d.index.names == ['i', 't', 'm', 'k'], f"Unexpected d index: {d.index.names}"

print(f"y observations : {len(y):,}")
print(f"d observations : {len(d):,}")

## Fit Regression Model

In [23]:
result = Regression(y=y, d=d)

In [24]:
result.predicted_expenditures()

### Predicted vs. actual log expenditures plot

In [25]:
%matplotlib inline

df_fit = pd.DataFrame({
    'y'   : y,
    'yhat': result.get_predicted_log_expenditures()
})

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(df_fit['yhat'], df_fit['y'], alpha=0.15, s=8, color='steelblue')

# 45-degree reference line
lims = [df_fit[['y', 'yhat']].min().min(), df_fit[['y', 'yhat']].max().max()]
ax.plot(lims, lims, 'r--', linewidth=1, label='45° line')

ax.set_xlabel('Predicted log expenditure ($\\hat{y}$)', fontsize=12)
ax.set_ylabel('Actual log expenditure ($y$)', fontsize=12)
ax.set_title('Predicted vs. Actual Log Expenditures — Tanzania', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

This graph shows only in-sample fit so it doesn't really help us that much. Could probably be improved by predicting fit on one wave using data from another. However, even with that, it wouldn't be useful since COVID-19 changed the outcomes.

## 4. Demand System [A]

### Frisch elasticities

The coefficients represent how income-elastic each food is. Higher values mean that a good's expenditure share rises more with total expenditure (it is more of a "luxury" within the food budget). Lower values indicates necessities that are always in demand regardless of price.

In [26]:
result.get_beta().sort_values()

In [27]:
result.graph_beta()
plt.show()

As you can see, the important staple foods that are crucial for survival are those in the bottom left of the graph with low elasticities: households need these goods no matter what.

On the opposite end of the graph, we see foods like citrus and other fruits which tend to be more expensive. Since these have higher Frisch elasticities, these are the first to go when income fluctuates and are often crowded out by the cheaper staple foods.

### 4.2 Household composition effects

These numbers summarize how household demographic composition shifts demand for each food, based on total expenditure. Each row is a food; each column is a demographic characteristic.

In [28]:
result.gamma

Here is the summarized household composition effects of demand, showing elasticities for each food based on age/sex group, also including a column for the effect of household size.

A few interesting trends stand out:
- Household size seems to be dominant driver in elasticity changes, having a much larger mean effect than any one age/sex group. This effect difference varies greatly between foods however, with the data suggesting that larger households tend to concentrate spending on staples rather than convenience foods.

- Older women (51+) show the strongest positive demographic effect on traditional staple foods, likely reflecting their role as the primary food decision-makers and a preference for traditional dietary patterns.

- Young children tend to suppress protein-rich and nutrient-dense food which is concerning.

- Teenage girls have the strongest positive demographic effect for most processed and convenience foods which is also concerning.

There seems to be systemic nutritional tensions between the people making food-decisions (working age men and older women) and the people who are most nutritionally vulnerable.

## Nutritional Content of Different Foods [B]

### Identify foods and nutrient mapping

In [29]:
import re
import time
import requests
import pandas as pd
import numpy as np

# Foods from the final CFE model
FOODS = [
    'Beef', 'Bread', 'Buns, Cakes And Biscuits', 'Cassava Fresh', 'Chicken',
    'Citrus Fruits', 'Coconuts', 'Cooking Oil', 'Eggs', 'Fish (dried)',
    'Fish (fresh)', 'Irish Potatoes', 'Leafy Greens', 'Macaroni, Spaghetti',
    'Maize (flour)', 'Milk (fresh)', 'Other Fruits', 'Other Spices',
    'Plantains', 'Pulses', 'Rice (husked)', 'Ripe Bananas', 'Salt',
    'Soft drinks', 'Sugar', 'Sweet Potatoes', 'Tea (dry)',
    'Vegetables (fresh)', 'Wheat Flour',
]

# Mapping nutrients to ids 
NUTRIENT_MAP = {
    'Energy':                          1008,  # kcal
    'Protein':                         1003,  # g
    'Fiber, total dietary':            1079,  # g
    'Folate, DFE':                     1190,  # µg
    'Calcium, Ca':                     1087,  # mg
    'Carbohydrate, by difference':     1005,  # g
    'Iron, Fe':                        1089,  # mg
    'Magnesium, Mg':                   1090,  # mg
    'Niacin':                          1167,  # mg
    'Phosphorus, P':                   1091,  # mg
    'Potassium, K':                    1092,  # mg
    'Riboflavin':                      1166,  # mg
    'Thiamin':                         1165,  # mg
    'Vitamin A, RAE':                  1106,  # µg
    'Vitamin B-12':                    1178,  # µg
    'Vitamin B-6':                     1175,  # µg
    'Vitamin C, total ascorbic acid':  1162,  # mg
    'Vitamin E (alpha-tocopherol)':    1109,  # mg
    'Vitamin K (phylloquinone)':       1185,  # µg
    'Zinc, Zn':                        1095,  # mg
}

FDC_API_KEY    = 'uOgTaIqGwW8Of1b6UcucKDg1yWcBt01FPea0hqDT'
FDC_SEARCH_URL = 'https://api.nal.usda.gov/fdc/v1/foods/search'
FDC_FOOD_URL   = 'https://api.nal.usda.gov/fdc/v1/food/{fdcId}'

### Find FDC IDs

In [30]:
# making searching the fdc database more accurate/efficient
SEARCH_OVERRIDES = {
    'Beef':                    'beef ground raw',
    'Bread':                   'bread white commercially prepared',
    'Buns, Cakes And Biscuits':'biscuits plain prepared from recipe',
    'Cassava Fresh':           'cassava raw',
    'Chicken':                 'chicken broilers or fryers meat only raw',
    'Citrus Fruits':           'oranges raw navels',
    'Coconuts':                'coconut meat raw',
    'Cooking Oil':             'oil vegetable sunflower',
    'Eggs':                    'egg whole raw fresh',
    'Fish (dried)':            'fish tilapia dried',
    'Fish (fresh)':            'fish tilapia raw',
    'Irish Potatoes':          'potatoes white flesh and skin raw',
    'Leafy Greens':            'spinach raw',
    'Macaroni, Spaghetti':     'pasta dry unenriched',
    'Maize (flour)':           'corn flour whole-grain yellow',
    'Milk (fresh)':            'milk whole 3.25% milkfat without added vitamin A and D',
    'Other Fruits':            'mixed fruit raw',
    'Other Spices':            'spices mixed',
    'Plantains':               'plantains green raw',
    'Pulses':                  'beans black mature seeds raw',
    'Rice (husked)':           'rice white long-grain regular raw unenriched',
    'Ripe Bananas':            'bananas raw',
    'Salt':                    'salt table',
    'Soft drinks':             'beverages carbonated cola regular',
    'Sugar':                   'sugars granulated',
    'Sweet Potatoes':          'sweet potato raw unprepared',
    'Tea (dry)':               'beverages tea black brewed prepared with tap water',
    'Vegetables (fresh)':      'tomatoes red ripe raw year round average',
    'Wheat Flour':             'wheat flour whole-grain',
}

def search_fdc(food_name, api_key, preferred_types=('Foundation', 'SR Legacy')):
    """Return the fdcId of the best FDC match for a food name, skipping any
    IDs that return 404 from the individual food endpoint."""
    query = SEARCH_OVERRIDES.get(food_name, food_name)
    # Strip characters the FDC API rejects and collapse whitespace (had issues with this causing errors eralier)
    query = re.sub(r'[(),]', ' ', query)
    query = re.sub(r'\s+', ' ', query).strip()

    params = {
        'query':    query,
        'dataType': ','.join(preferred_types),
        'pageSize': 5,
        'api_key':  api_key,
    }
    resp = requests.get(FDC_SEARCH_URL, params=params)
    resp.raise_for_status()
    candidates = resp.json().get('foods', [])

    for candidate in candidates:
        fdc_id = candidate['fdcId']
        check = requests.get(FDC_FOOD_URL.format(fdcId=fdc_id), params={'api_key': api_key})
        if check.status_code == 200:
            return fdc_id, candidate['description']
        elif check.status_code == 404:
            print(f"  [{food_name}] fdcId {fdc_id} returned 404, trying next candidate...")
            time.sleep(0.2)
            continue
        else:
            check.raise_for_status()

    return None, None 

# Search for all 29 foods
food_ids = {}
print(f"{'Food':<30} {'fdcId':>8}  Description")
print('-' * 80)
for food in FOODS:
    fdc_id, desc = search_fdc(food, FDC_API_KEY)
    food_ids[food] = fdc_id
    print(f"{food:<30} {str(fdc_id):>8}  {desc}")
    time.sleep(0.2)

In [32]:
# final food_df with nutrients
def fetch_nutrients(fdc_id, nutrient_ids, api_key):
    """Return a dict {nutrient_name: value_per_100g} for the requested nutrient IDs."""
    url = FDC_FOOD_URL.format(fdcId=fdc_id)
    resp = requests.get(url, params={'api_key': api_key})
    resp.raise_for_status()
    data = resp.json()

    # Build a lookup: nutrient_id -> value
    id_to_value = {}
    for n in data.get('foodNutrients', []):
        nid = (n.get('nutrient') or {}).get('id') or n.get('nutrientId')
        val = n.get('amount') or n.get('value')
        if nid and val is not None:
            id_to_value[nid] = val

    return {name: id_to_value.get(nid) for name, nid in nutrient_ids.items()}

# Fetch nutrient data for every food
records = []
for food, fdc_id in food_ids.items():
    if fdc_id is None:
        print(f"WARNING: no FDC match found for '{food}', skipping.")
        continue
    nutrients = fetch_nutrients(fdc_id, NUTRIENT_MAP, FDC_API_KEY)
    nutrients['food']  = food
    nutrients['fdcId'] = fdc_id
    records.append(nutrients)
    time.sleep(0.2) # had issues with rate of requests

food_df = pd.DataFrame(records).set_index('food')
print(food_df.shape)
food_df

In [33]:
# Diet minimums
rdi = pd.read_csv('diet_minimums.csv', index_col=0)
rdi = rdi.set_index('Nutrition').drop(columns='Source')

# Show for F 19-30 and M 19-30
rdi_adult = rdi[['F 19-30', 'M 19-30']].apply(pd.to_numeric, errors='coerce').mean(axis=1)
rdi_adult.name = 'RDI_adult_avg'

# Nutrients per 100g
nutrient_cols  = list(NUTRIENT_MAP.keys())
food_nutrients = food_df[nutrient_cols].apply(pd.to_numeric, errors='coerce')

# % of adult RDI provided per 100g of each food
coverage_pct = food_nutrients.div(rdi_adult, axis=1) * 100
coverage_pct.columns.name = 'nutrient'
coverage_pct.index.name   = 'food'

print("% of adult RDI provided per 100g (for selected nutrients)")
display_cols = ['Energy', 'Protein', 'Iron, Fe', 'Calcium, Ca',
                'Vitamin A, RAE', 'Vitamin C, total ascorbic acid', 'Zinc, Zn']
print(coverage_pct[display_cols].round(1).to_string())

In [34]:
# nutritional gaps and heatmap
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Best single food for each nutrient across the whole basket
max_coverage = coverage_pct.max(axis=0)

gaps   = max_coverage[max_coverage <  20].sort_values()
strong = max_coverage[max_coverage >= 20].sort_values(ascending=False)

print("Nutrients with WEAK coverage (no single food provides ≥20% RDI per 100g):")
print(gaps.round(1).to_string())
print("\nNutrients with GOOD coverage (≥20% RDI from at least one food):")
print(strong.round(1).to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(14, 9))
data = coverage_pct[nutrient_cols].clip(upper=100)

im = ax.imshow(data.values, aspect='auto', cmap='RdYlGn',
               norm=mcolors.Normalize(vmin=0, vmax=100))

ax.set_xticks(range(len(nutrient_cols)))
ax.set_xticklabels(nutrient_cols, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(data)))
ax.set_yticklabels(data.index, fontsize=8)
ax.set_title(
    '% of Adult RDI per 100g — Tanzania food basket\n(capped at 100%; green = meets RDI)',
    fontsize=12
)
plt.colorbar(im, ax=ax, label='% of RDI')
plt.tight_layout()
plt.savefig('tanzania_nutrient_coverage.png', dpi=150)
plt.show()

**Explanation:** While this heatmap is important to understand the landscape of foods and their nutritional content, it can be misleading. First of all, the 29 foods that survived to this point in the model are chosen because they have enough observations to be relevant, so more nutrient-dense foods could be excluded. Secondly, no singly food is meant to cover then entire RDI for a nutrient...a diet is supposed to include many of these foods.

## Nutritional Adequacy at the household level [B]

In [35]:
# Load prices w/both waves, take prices
p1 = pd.read_excel('Tanzania.xlsx', sheet_name='Food Prices (2019-20)')
p2 = pd.read_excel('Tanzania.xlsx', sheet_name='Food Prices (2020-21)')
prices = pd.concat([p1, p2], ignore_index=True)

# Convert prices to per gram
prices['price_per_g'] = prices['Price'] / 1000
price_map = (prices.groupby(['t', 'm', 'j'])['price_per_g']
                   .median()
                   .reset_index()
                   .set_index(['t', 'm', 'j'])['price_per_g'])

# Load food expenditures
x1 = pd.read_excel('Tanzania.xlsx', sheet_name='Food Expenditures (2019-20)')
x2 = pd.read_excel('Tanzania.xlsx', sheet_name='Food Expenditures (2020-21)')
x_long = pd.concat([x1, x2], ignore_index=True)
for col in ['i', 't', 'm']:
    x_long[col] = x_long[col].astype(str)

# Keep only the 29 foods in the model (data consistency)
x_long = x_long[x_long['j'].isin(FOODS)]

x_long = x_long.join(price_map, on=['t', 'm', 'j'])

# If price is missing, use median
national_price = prices.groupby('j')['price_per_g'].median()
x_long['price_per_g'] = x_long['price_per_g'].fillna(x_long['j'].map(national_price))

# Grams purchased per recall period
x_long['grams'] = x_long['Expenditure'] / x_long['price_per_g']

# The survey recall period is 7 days —> convert to grams/day
RECALL_DAYS = 7
x_long['grams_per_day'] = x_long['grams'] / RECALL_DAYS

### Daily nutrient intake per household

In [36]:
# Nutrient content per gram (food_df is per 100g)
nutrients_per_g = food_nutrients / 100

# Map nutrient content onto expenditure rows
nutrient_cols = list(NUTRIENT_MAP.keys())
for col in nutrient_cols:
    x_long[col] = x_long['j'].map(nutrients_per_g[col])

# Nutrient intake per food per day = grams_per_day * nutrient_per_gram
intake_cols = []
for col in nutrient_cols:
    intake_col = f'intake_{col}'
    x_long[intake_col] = x_long['grams_per_day'] * x_long[col]
    intake_cols.append(intake_col)

# Get total daily nutrient intake per household
hh_intake = (x_long.groupby(['i', 't', 'm'])[intake_cols]
                    .sum()
                    .rename(columns=lambda c: c.replace('intake_', '')))

print(f"Households with valid nutrient intake estimates: {len(hh_intake):,}")
hh_intake.head()

### Household specific RDI targets

In [37]:
# Reload household characteristics (filtered to our two waves)
d_raw = pd.read_excel('Tanzania.xlsx', sheet_name='Household Characteristics')
d_hh = d_raw[d_raw['t'].isin(['2019-20', '2020-21'])].copy()
for col in ['i', 't', 'm']:
    d_hh[col] = d_hh[col].astype(str)
d_hh = d_hh.set_index(['i', 't', 'm']).fillna(0)


rdi = pd.read_csv('diet_minimums.csv', index_col=0)
rdi = rdi.set_index('Nutrition').drop(columns='Source')
rdi = rdi.apply(pd.to_numeric, errors='coerce')

# Map demographic columns in d_hh to RDI columns
DEMO_TO_RDI = {
    'Males 00-03':   'C 1-3',
    'Females 00-03': 'C 1-3',
    'Males 04-08':   'M 4-8',
    'Females 04-08': 'F 4-8',
    'Males 09-13':   'M 9-13',
    'Females 09-13': 'F 9-13',
    'Males 14-18':   'M 14-18',
    'Females 14-18': 'F 14-18',
    'Males 19-30':   'M 19-30',
    'Females 19-30': 'F 19-30',
    'Males 31-50':   'M 31-50',
    'Females 31-50': 'F 31-50',
    'Males 51-99':   'M 51+',
    'Females 51-99': 'F 51+',
}

# For each household, RDI_hh = sum over members of (n_members_in_group * RDI_for_group)
demo_cols = [c for c in DEMO_TO_RDI if c in d_hh.columns]

rdi_hh = pd.DataFrame(index=d_hh.index, columns=nutrient_cols, dtype=float)
rdi_hh[:] = 0.0

for demo_col in demo_cols:
    rdi_col = DEMO_TO_RDI[demo_col]
    if rdi_col not in rdi.columns:
        continue
    # n_members in this group for each household (shape: n_households)
    n_members = d_hh[demo_col]
    # RDI for this group for each nutrient (shape: n_nutrients)
    rdi_for_group = rdi[rdi_col]
    # Outer product: each household's contribution from this demographic group
    contribution = pd.DataFrame(
        np.outer(n_members.values, rdi_for_group.values),
        index=n_members.index,
        columns=rdi_for_group.index
    )
    rdi_hh = rdi_hh.add(contribution.reindex(columns=nutrient_cols), fill_value=0)

rdi_hh = rdi_hh.apply(pd.to_numeric)
print(f"Household RDI targets computed for {len(rdi_hh):,} households")
rdi_hh.head()

### Compare intake to RDI and evaluate adequacy

In [38]:
# Align households with expenditure and demographic data
common_idx = hh_intake.index.intersection(rdi_hh.index)
intake_aligned = hh_intake.loc[common_idx, nutrient_cols]
rdi_aligned    = rdi_hh.loc[common_idx, nutrient_cols]

print(f"Households in joint sample: {len(common_idx):,}")

# Adequacy ratio: intake / household RDI (>1 means household meets its target)
adequacy = intake_aligned / rdi_aligned

# Proportion of households meeting RDI for each nutrient
meets_rdi = (adequacy >= 1.0).mean() * 100  # percent

print("% of households meeting household RDI")
print(meets_rdi.sort_values().round(1).to_string())

In [39]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: bar chart of % households meeting RDI
meets_sorted = meets_rdi.sort_values()
colors = ['#d73027' if v < 50 else '#fee08b' if v < 75 else '#1a9850'
          for v in meets_sorted]
axes[0].barh(meets_sorted.index, meets_sorted.values, color=colors)
axes[0].axvline(50, color='black', linewidth=0.8, linestyle='--')
axes[0].axvline(75, color='black', linewidth=0.8, linestyle=':')
axes[0].set_xlabel('% of households meeting household RDI')
axes[0].set_title('Dietary adequacy by nutrient\n(red <50%, yellow 50–75%, green ≥75%)')
axes[0].set_xlim(0, 100)

# Right: heatmap of median adequacy ratio by nutrient and survey wave
adequacy_wave = adequacy.reset_index()
med_by_wave = (adequacy_wave.groupby('t')[nutrient_cols]
                             .median()
                             .clip(upper=2))

im = axes[1].imshow(med_by_wave.values, aspect='auto', cmap='RdYlGn',
                    norm=mcolors.Normalize(vmin=0, vmax=2))
axes[1].set_xticks(range(len(nutrient_cols)))
axes[1].set_xticklabels(nutrient_cols, rotation=45, ha='right', fontsize=8)
axes[1].set_yticks(range(len(med_by_wave)))
axes[1].set_yticklabels(med_by_wave.index)
axes[1].set_title('Median adequacy ratio by wave\n(1.0 = meets RDI; capped at 2×)')
plt.colorbar(im, ax=axes[1], label='Intake / RDI')

# Summary table
summary = pd.DataFrame({
    'pct_meeting_RDI':    meets_rdi.round(1),
    'median_adequacy':    adequacy.median().round(2),
    'pct_below_half_RDI': ((adequacy < 0.5).mean() * 100).round(1),
}).sort_values('pct_meeting_RDI')

print("Adequacy summary")
print(summary.to_string())

Here we see the % of Households that actually meet the recommended dietary intakes that we outlined earlier. The graph shows the percentage of households meeting the household RDI on the x-axis with the specific nutrient on the y-axis. The exact values are shown in the table in the top right.

The heatmap shows not just whether households meet requirements, but how far they are from them. An interesting trend appears in that several RDIs move from slightly green to yellowish from the first wave to the second. You would expect nutritional outcomes to increase over time, not regress. This is very likely due to the COVID-19 pandemic. From 2019-2020, COVID had not yet reached Tanzania; but in the subsequent years, the pandemic had a huge impact: disrupting supply chains, the tourism industry, and dramatically lowering income and purchasing power.

### Nutrient Deficiencies
Calcium is a catastrophic deficiency and should be our main focus. Only 2.2% of households meet this RDI with 87% below half their requirement (nothing else comes close). The food basket represented by our 29 foods has no adequate calcium source other than milk (which has a higher Frisch elasticity of 0.52 so it is not within budget for many households).

Vitamin A is the next most severe deficiency. Only 12% meet the RDI, with 69% below half of the RDI. Sweet potatoes and leafy greens are the main sources of Vitamin A in the food basket but both have higher Frisch elasticities (0.63 and 0.70 respectively).

Potassium and B-12 also represent severe deficiences with only 13% of householdes meeting the potassium RDI and 17.2% meeting B-12. The B-12 deficiency is likely explained by the low animal food consumption levels which also drives the zinc and iron deficiencies. 

Riboflavin, Vitamin C, Niacin, Vitamin E, and Energy make up the middle tier where roughly 23–44% of households meet requirements. Energy itself is only reahced by 43.9% of households so generally, caloric needs are not being met.

### Policy Goals

Goal 1: Calcium – Reduce the number of households below 50% RDI from 87% to around 65%. Realistically, we could fortify a widely-consumed staple food such as maize flour. Income transfers won’t work on their own.

Goal 2: Vitamin A – Reduce the number of households below 75% RDI from ~88% to below 65%. Fortification of cooking oil is the best intervention and is already in place but compliance is the issue. Income transfers/growth could also help because leafy greens and sweet potatoes are available but have higher Frisch elasticities.

Goal 3: Vitamin B-12 & Potassium – While goals 1 and 2 should be the biggest focus, these are also worth addressing. B-12 requires animal-source food subsidies or direct supplementation. Potassium can be addressed through increased fruit and vegetable consumption which again can be increased through income transfers and subsidies/incentives.

Iron and Zinc not primary targets. Caloric adequacy (energy) is mixed, median household almost reaches RDI but only 43.9% totally meet it. Simply addressing nutrient deficiencies cannot be totally separate from overall food access.

## Policy Simulations

**NOTE**: Policy simulations are conducted using model-predicted rather than actual expenditures for two reasons. First, actual survey expenditures reflect a single 7-day recall period, meaning many households record zero purchases for foods they regularly consume but happened not to buy that week. Using these zeros as a baseline would understate true consumption and make policies appear more effective than they are by artificially depressing the starting expenditure point. Second, and more fundamentally, consistency requires that the baseline and the counterfactual be generated by the same model — applying demand elasticities estimated from the CFE regression to actual expenditures would mix two different data-generating processes. Predicted expenditures represent each household's expected demand given their income, demographics, and market prices, which is the correct baseline against which to measure a structural change in those same variables. Policy impacts are therefore reported as percentage-point changes relative to the predicted baseline rather than as absolute adequacy rates, which should be taken from the observed-expenditure analysis.

In [55]:
# Cell — Shared constants (run before any simulation chunks)

TARGET_NUTRIENTS = [
    'Vitamin A, RAE',
    'Calcium, Ca',
    'Iron, Fe',
    'Zinc, Zn',
    'Vitamin B-12',
    'Potassium, K',
]

FOODS = [
    'Beef', 'Bread', 'Buns, Cakes And Biscuits', 'Cassava Fresh', 'Chicken',
    'Citrus Fruits', 'Coconuts', 'Cooking Oil', 'Eggs', 'Fish (dried)',
    'Fish (fresh)', 'Irish Potatoes', 'Leafy Greens', 'Macaroni, Spaghetti',
    'Maize (flour)', 'Milk (fresh)', 'Other Fruits', 'Other Spices',
    'Plantains', 'Pulses', 'Rice (husked)', 'Ripe Bananas', 'Salt',
    'Soft drinks', 'Sugar', 'Sweet Potatoes', 'Tea (dry)',
    'Vegetables (fresh)', 'Wheat Flour',
]

nutrient_per_100g = {
    'Beef':                    {'Vitamin A, RAE': 0,    'Calcium, Ca': 12,  'Iron, Fe': 2.6,  'Zinc, Zn': 4.8,  'Vitamin B-12': 2.60,  'Potassium, K': 318},
    'Bread':                   {'Vitamin A, RAE': 0,    'Calcium, Ca': 151, 'Iron, Fe': 3.6,  'Zinc, Zn': 0.7,  'Vitamin B-12': 0.01,  'Potassium, K': 115},
    'Buns, Cakes And Biscuits':{'Vitamin A, RAE': 15,   'Calcium, Ca': 141, 'Iron, Fe': 1.8,  'Zinc, Zn': 0.5,  'Vitamin B-12': 0.10,  'Potassium, K': 120},
    'Cassava Fresh':           {'Vitamin A, RAE': 1,    'Calcium, Ca': 16,  'Iron, Fe': 0.3,  'Zinc, Zn': 0.3,  'Vitamin B-12': 0.00,  'Potassium, K': 271},
    'Chicken':                 {'Vitamin A, RAE': 9,    'Calcium, Ca': 11,  'Iron, Fe': 0.9,  'Zinc, Zn': 1.3,  'Vitamin B-12': 0.31,  'Potassium, K': 223},
    'Citrus Fruits':           {'Vitamin A, RAE': 11,   'Calcium, Ca': 40,  'Iron, Fe': 0.1,  'Zinc, Zn': 0.1,  'Vitamin B-12': 0.00,  'Potassium, K': 181},
    'Coconuts':                {'Vitamin A, RAE': 0,    'Calcium, Ca': 14,  'Iron, Fe': 2.4,  'Zinc, Zn': 1.1,  'Vitamin B-12': 0.00,  'Potassium, K': 356},
    'Cooking Oil':             {'Vitamin A, RAE': 0,    'Calcium, Ca': 0,   'Iron, Fe': 0.0,  'Zinc, Zn': 0.0,  'Vitamin B-12': 0.00,  'Potassium, K': 0},
    'Eggs':                    {'Vitamin A, RAE': 160,  'Calcium, Ca': 56,  'Iron, Fe': 1.8,  'Zinc, Zn': 1.3,  'Vitamin B-12': 0.89,  'Potassium, K': 138},
    'Fish (dried)':            {'Vitamin A, RAE': 15,   'Calcium, Ca': 170, 'Iron, Fe': 2.9,  'Zinc, Zn': 1.5,  'Vitamin B-12': 2.80,  'Potassium, K': 340},
    'Fish (fresh)':            {'Vitamin A, RAE': 8,    'Calcium, Ca': 14,  'Iron, Fe': 0.4,  'Zinc, Zn': 0.4,  'Vitamin B-12': 1.58,  'Potassium, K': 302},
    'Irish Potatoes':          {'Vitamin A, RAE': 0,    'Calcium, Ca': 9,   'Iron, Fe': 0.5,  'Zinc, Zn': 0.3,  'Vitamin B-12': 0.00,  'Potassium, K': 421},
    'Leafy Greens':            {'Vitamin A, RAE': 469,  'Calcium, Ca': 99,  'Iron, Fe': 2.7,  'Zinc, Zn': 0.5,  'Vitamin B-12': 0.00,  'Potassium, K': 558},
    'Macaroni, Spaghetti':     {'Vitamin A, RAE': 0,    'Calcium, Ca': 7,   'Iron, Fe': 0.5,  'Zinc, Zn': 0.5,  'Vitamin B-12': 0.00,  'Potassium, K': 44},
    'Maize (flour)':           {'Vitamin A, RAE': 0,    'Calcium, Ca': 4,   'Iron, Fe': 1.0,  'Zinc, Zn': 0.6,  'Vitamin B-12': 0.00,  'Potassium, K': 142},
    'Milk (fresh)':            {'Vitamin A, RAE': 46,   'Calcium, Ca': 113, 'Iron, Fe': 0.0,  'Zinc, Zn': 0.4,  'Vitamin B-12': 0.45,  'Potassium, K': 150},
    'Other Fruits':            {'Vitamin A, RAE': 20,   'Calcium, Ca': 15,  'Iron, Fe': 0.4,  'Zinc, Zn': 0.1,  'Vitamin B-12': 0.00,  'Potassium, K': 200},
    'Other Spices':            {'Vitamin A, RAE': 5,    'Calcium, Ca': 50,  'Iron, Fe': 1.0,  'Zinc, Zn': 0.5,  'Vitamin B-12': 0.00,  'Potassium, K': 100},
    'Plantains':               {'Vitamin A, RAE': 56,   'Calcium, Ca': 3,   'Iron, Fe': 0.6,  'Zinc, Zn': 0.1,  'Vitamin B-12': 0.00,  'Potassium, K': 499},
    'Pulses':                  {'Vitamin A, RAE': 0,    'Calcium, Ca': 123, 'Iron, Fe': 5.0,  'Zinc, Zn': 3.7,  'Vitamin B-12': 0.00,  'Potassium, K': 1483},
    'Rice (husked)':           {'Vitamin A, RAE': 0,    'Calcium, Ca': 9,   'Iron, Fe': 0.3,  'Zinc, Zn': 0.5,  'Vitamin B-12': 0.00,  'Potassium, K': 35},
    'Ripe Bananas':            {'Vitamin A, RAE': 3,    'Calcium, Ca': 5,   'Iron, Fe': 0.3,  'Zinc, Zn': 0.2,  'Vitamin B-12': 0.00,  'Potassium, K': 358},
    'Salt':                    {'Vitamin A, RAE': 0,    'Calcium, Ca': 24,  'Iron, Fe': 0.3,  'Zinc, Zn': 0.1,  'Vitamin B-12': 0.00,  'Potassium, K': 8},
    'Soft drinks':             {'Vitamin A, RAE': 0,    'Calcium, Ca': 3,   'Iron, Fe': 0.0,  'Zinc, Zn': 0.0,  'Vitamin B-12': 0.00,  'Potassium, K': 2},
    'Sugar':                   {'Vitamin A, RAE': 0,    'Calcium, Ca': 1,   'Iron, Fe': 0.1,  'Zinc, Zn': 0.0,  'Vitamin B-12': 0.00,  'Potassium, K': 2},
    'Sweet Potatoes':          {'Vitamin A, RAE': 961,  'Calcium, Ca': 30,  'Iron, Fe': 0.6,  'Zinc, Zn': 0.3,  'Vitamin B-12': 0.00,  'Potassium, K': 337},
    'Tea (dry)':               {'Vitamin A, RAE': 0,    'Calcium, Ca': 0,   'Iron, Fe': 0.0,  'Zinc, Zn': 0.0,  'Vitamin B-12': 0.00,  'Potassium, K': 37},
    'Vegetables (fresh)':      {'Vitamin A, RAE': 42,   'Calcium, Ca': 10,  'Iron, Fe': 0.3,  'Zinc, Zn': 0.2,  'Vitamin B-12': 0.00,  'Potassium, K': 237},
    'Wheat Flour':             {'Vitamin A, RAE': 0,    'Calcium, Ca': 15,  'Iron, Fe': 3.9,  'Zinc, Zn': 2.6,  'Vitamin B-12': 0.00,  'Potassium, K': 107},
}

nutrients_df = pd.DataFrame(nutrient_per_100g).T

# ── Prices ────────────────────────────────────────────────────────────────────
p1 = pd.read_excel('Tanzania.xlsx', sheet_name='Food Prices (2019-20)')
p2 = pd.read_excel('Tanzania.xlsx', sheet_name='Food Prices (2020-21)')
prices = pd.concat([p1, p2], ignore_index=True)
prices['t'] = prices['t'].astype(str)
prices['m'] = prices['m'].astype(str)
prices['price_per_g'] = prices['Price'] / 1000
price_map      = prices.groupby(['t', 'm', 'j'])['price_per_g'].median()
national_price = prices.groupby('j')['price_per_g'].median()

# ── Household RDI targets ─────────────────────────────────────────────────────
rdi_raw = pd.read_csv('diet_minimums.csv', index_col=0)
rdi     = rdi_raw.set_index('Nutrition').drop(columns='Source').apply(pd.to_numeric, errors='coerce')

DEMO_TO_RDI = {
    'Males 00-03':   'C 1-3',  'Females 00-03': 'C 1-3',
    'Males 04-08':   'M 4-8',  'Females 04-08': 'F 4-8',
    'Males 09-13':   'M 9-13', 'Females 09-13': 'F 9-13',
    'Males 14-18':   'M 14-18','Females 14-18': 'F 14-18',
    'Males 19-30':   'M 19-30','Females 19-30': 'F 19-30',
    'Males 31-50':   'M 31-50','Females 31-50': 'F 31-50',
    'Males 51-99':   'M 51+',  'Females 51-99': 'F 51+',
}

d_hh = pd.read_excel('Tanzania.xlsx', sheet_name='Household Characteristics')
d_hh = d_hh[d_hh['t'].isin(['2019-20', '2020-21'])].copy()
for col in ['i', 't', 'm']:
    d_hh[col] = d_hh[col].astype(str)
d_hh = d_hh.set_index(['i', 't', 'm']).fillna(0)

rdi_hh = pd.DataFrame(0.0, index=d_hh.index, columns=TARGET_NUTRIENTS)
for demo_col, rdi_col in DEMO_TO_RDI.items():
    if demo_col not in d_hh.columns or rdi_col not in rdi.columns:
        continue
    for nut in TARGET_NUTRIENTS:
        if nut in rdi.index:
            rdi_hh[nut] += d_hh[demo_col] * rdi.loc[nut, rdi_col]

# ── pe_long: predicted expenditures in long format with prices attached ───────
# (requires result object from the demand estimation cells above)
pe = result.predicted_expenditures()
pe_long = pe.reset_index()
pe_long.columns = ['i', 't', 'm', 'j', 'expenditure']
pe_long = pe_long[pe_long['j'].isin(FOODS)].copy()
pe_long = pe_long.join(price_map, on=['t', 'm', 'j'])
pe_long['price_per_g'] = pe_long['price_per_g'].fillna(pe_long['j'].map(national_price))

In [54]:
# Simulation baseline: use predicted expenditures from demand system
# This is the right baseline for counterfactuals because: it covers all foods for all households via the estimated demand curves, 
# it smooths over the recall-period zero problem in raw expenditure data, counterfactual changes are applied through the same model

pe = result.predicted_expenditures()  # TZS/week per household per food
beta = result.get_beta()              # Frisch elasticities
w = result.get_w()                 # log total expenditure index per household

# Reshape predicted expenditures into a wide dataframe (households x foods)
pe_wide = pe.unstack('j').reindex(columns=FOODS)

# Attach prices to convert expenditures -> grams/day
pe_long = pe.reset_index()
pe_long.columns = ['i','t','m','j','expenditure']
pe_long = pe_long[pe_long['j'].isin(FOODS)].copy()
pe_long = pe_long.join(price_map, on=['t','m','j'])
pe_long['price_per_g'] = pe_long['price_per_g'].fillna(pe_long['j'].map(national_price))
pe_long['grams_per_day'] = pe_long['expenditure'] / pe_long['price_per_g'] / 7

# Baseline nutrient intake per household per day
for nut in TARGET_NUTRIENTS:
    pe_long[nut] = pe_long['j'].map(nutrients_df[nut]) * pe_long['grams_per_day'] / 100

hh_intake_baseline = pe_long.groupby(['i','t','m'])[TARGET_NUTRIENTS].sum()

# Baseline adequacy ratios
common = hh_intake_baseline.index.intersection(rdi_hh.index)
baseline_adequacy = hh_intake_baseline.loc[common] / rdi_hh.loc[common]

def adequacy_summary(adequacy_df, label=''):
    """Print and return a summary Series of % meeting RDI per nutrient."""
    pct = (adequacy_df >= 1.0).mean() * 100
    if label:
        print(f"\n{'='*55}")
        print(f"  {label}")
        print(f"{'='*55}")
        for nut in TARGET_NUTRIENTS:
            print(f"  {nut:<25} {pct[nut]:5.1f}% meeting RDI")
    return pct

baseline_pct = adequacy_summary(baseline_adequacy, 'BASELINE')

In [56]:
def simulate(pe_long_in, nutrient_content=None, label='Simulation'):
    """
    Run a counterfactual and return adequacy rates.

    Parameters
    ----------
    pe_long_in : DataFrame
        Long-format predicted expenditures with columns
        [i, t, m, j, expenditure, price_per_g, grams_per_day].
        Pass a modified copy for each simulation.
    nutrient_content : dict or None
        Override nutrient content per 100g for specific foods.
        e.g. {'Cooking Oil': {'Vitamin A, RAE': 800}}
        If None, uses the original nutrients_df.
    label : str
        Label for printed output.

    Returns
    -------
    adequacy_df : DataFrame of adequacy ratios (households x nutrients)
    pct_meeting : Series of % households meeting RDI per nutrient
    """
    nut_df = nutrients_df.copy()
    if nutrient_content:
        for food, updates in nutrient_content.items():
            for nut, val in updates.items():
                nut_df.loc[food, nut] = val

    sim = pe_long_in.copy()
    sim['grams_per_day'] = sim['expenditure'] / sim['price_per_g'] / 7

    for nut in TARGET_NUTRIENTS:
        sim[nut] = sim['j'].map(nut_df[nut]) * sim['grams_per_day'] / 100

    hh_intake = sim.groupby(['i','t','m'])[TARGET_NUTRIENTS].sum()
    common_idx = hh_intake.index.intersection(rdi_hh.index)
    adequacy   = hh_intake.loc[common_idx] / rdi_hh.loc[common_idx]
    pct        = adequacy_summary(adequacy, label)
    return adequacy, pct

In [57]:
# Policy 1: Cash transfer equal to 20% of median weekly food expenditure 
# Mechanism: shifting w (log total expenditure) shifts log demand for each food
# by beta_j * delta_w. This is the core CFE demand response.
#
# We simulate three transfer sizes to show dose-response:
#   Small  = 10% income increase
#   Medium = 20% income increase  (headline policy)
#   Large  = 50% income increase

def apply_income_shock(pe_long_base, beta, delta_w_scalar):
    """
    Scale predicted expenditures by exp(beta_j * delta_w) for each food.
    delta_w_scalar is a single float applied uniformly to all households.
    """
    sim = pe_long_base.copy()
    scale = sim['j'].map(beta).apply(lambda b: np.exp(b * delta_w_scalar))
    sim['expenditure'] = sim['expenditure'] * scale
    return sim

transfer_sizes = {
    'Transfer +10%': np.log(1.10),
    'Transfer +20%': np.log(1.20),
    'Transfer +50%': np.log(1.50),
}

income_results = {'Baseline': baseline_pct}
for label, delta_w in transfer_sizes.items():
    sim_data = apply_income_shock(pe_long, beta, delta_w)
    _, pct = simulate(sim_data, label=label)
    income_results[label] = pct

# Summary table
income_summary = pd.DataFrame(income_results).T[TARGET_NUTRIENTS]
print("\n=== INCOME TRANSFER SUMMARY (% meeting RDI) ===")
print(income_summary.round(1).to_string())

In [58]:
# Policy 2: Price subsidy on the three highest Vitamin A and Calcium foods
# Mechanism: a subsidy reduces effective price_per_g, increasing grams_per_day
# for a given level of nominal expenditure.
# We assume own-price elasticity of -0.5 (conservative, mid-range for staples
# in Sub-Saharan Africa literature) to scale the expenditure response.
#
# Target foods chosen by nutrient density per TZS:
#   Vitamin A: Sweet Potatoes, Leafy Greens, Eggs
#   Calcium:   Dried Fish, Bread, Milk (fresh)

OWN_PRICE_ELASTICITY = -0.5  # assumed; note this as a limitation

def apply_price_subsidy(pe_long_base, price_map_base, subsidy_foods, subsidy_rate,
                        elasticity=OWN_PRICE_ELASTICITY):
    """
    Apply a price subsidy to specific foods.

    Parameters
    ----------
    subsidy_foods : list of food names to subsidise
    subsidy_rate  : fraction reduction in price (e.g. 0.30 = 30% cheaper)
    elasticity    : own-price elasticity (negative)
    """
    sim = pe_long_base.copy()
    # For subsidised foods: new price = old price * (1 - subsidy_rate)
    # Quantity response: delta_q/q = elasticity * delta_p/p = elasticity * (-subsidy_rate)
    # => new expenditure = old_exp * (1 - subsidy_rate) * (1 + |elasticity| * subsidy_rate)
    mask = sim['j'].isin(subsidy_foods)
    price_effect    = (1 - subsidy_rate)
    quantity_effect = (1 + abs(elasticity) * subsidy_rate)
    sim.loc[mask, 'price_per_g']  = sim.loc[mask, 'price_per_g'] * price_effect
    sim.loc[mask, 'expenditure']  = sim.loc[mask, 'expenditure'] * price_effect * quantity_effect
    return sim

# Vitamin A subsidy: 30% price reduction on sweet potatoes, leafy greens, eggs
vit_a_foods   = ['Sweet Potatoes', 'Leafy Greens', 'Eggs']
# Calcium subsidy: 30% price reduction on dried fish, bread, milk
calcium_foods = ['Fish (dried)', 'Bread', 'Milk (fresh)']
# Combined subsidy on all six foods
combined_foods = vit_a_foods + calcium_foods

subsidy_scenarios = {
    'Subsidy: Vit A foods 30%':   (vit_a_foods,   0.30),
    'Subsidy: Calcium foods 30%': (calcium_foods,  0.30),
    'Subsidy: All targets 30%':   (combined_foods, 0.30),
    'Subsidy: All targets 50%':   (combined_foods, 0.50),
}

subsidy_results = {'Baseline': baseline_pct}
for label, (foods, rate) in subsidy_scenarios.items():
    sim_data = apply_price_subsidy(pe_long, price_map, foods, rate)
    _, pct = simulate(sim_data, label=label)
    subsidy_results[label] = pct

subsidy_summary = pd.DataFrame(subsidy_results).T[TARGET_NUTRIENTS]
print("\n=== PRICE SUBSIDY SUMMARY (% meeting RDI) ===")
print(subsidy_summary.round(1).to_string())

In [59]:
# Policy 3: Fortification
# Mechanism: fortification changes nutrient content per 100g without changing
# quantities purchased. No demand response — pure nutritional accounting.
# This is the correct model for a mandatory fortification program.
#
# Scenarios based on Tanzania's actual fortification program:
#   (a) Cooking oil: Vitamin A fortified to 750 µg RAE / 100g (WHO standard)
#   (b) Maize flour: fortified to 100 µg Vitamin A and 200mg Calcium / 100g
#       (realistic for double-fortified flour)
#   (c) Full compliance: both oil and maize at WHO standards

# Note: current Cooking Oil Vit A in fdc_df is ~0 (unfortified baseline)
# Tanzania standard for fortified oil: 750 µg RAE / 100g (edible oil standard)
# Maize flour standard: iron 60mg, Vit A 100µg, Calcium not typically fortified
# We use conservative estimates to avoid overstating impact

fortification_scenarios = {
    'Fortify: Oil (Vit A)': {
        'Cooking Oil': {'Vitamin A, RAE': 750}
    },
    'Fortify: Maize flour (Vit A + Ca)': {
        'Maize (flour)': {'Vitamin A, RAE': 100, 'Calcium, Ca': 200}
    },
    'Fortify: Full compliance (oil + maize)': {
        'Cooking Oil':  {'Vitamin A, RAE': 750},
        'Maize (flour)': {'Vitamin A, RAE': 100, 'Calcium, Ca': 200}
    },
    'Fortify: Full + Wheat flour (Ca)': {
        'Cooking Oil':   {'Vitamin A, RAE': 750},
        'Maize (flour)': {'Vitamin A, RAE': 100, 'Calcium, Ca': 200},
        'Wheat Flour':   {'Calcium, Ca': 200}
    },
}

fortification_results = {'Baseline': baseline_pct}
for label, nutrient_overrides in fortification_scenarios.items():
    # Quantities unchanged — pass original pe_long, only change nutrient content
    _, pct = simulate(pe_long, nutrient_content=nutrient_overrides, label=label)
    fortification_results[label] = pct

fortification_summary = pd.DataFrame(fortification_results).T[TARGET_NUTRIENTS]
print("\n=== FORTIFICATION SUMMARY (% meeting RDI) ===")
print(fortification_summary.round(1).to_string())

In [66]:
# ── Figure 1: Policy comparison bar chart per nutrient ───────────────────────
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

POLICY_COLORS = {
    'Baseline':                               '#888888',
    'Transfer +10%':                          '#b3cde3',
    'Transfer +20%':                          '#6baed6',
    'Transfer +50%':                          '#2171b5',
    'Subsidy: Vit A foods 30%':               '#fed976',
    'Subsidy: Calcium foods 30%':             '#fd8d3c',
    'Subsidy: All targets 30%':               '#e31a1c',
    'Subsidy: All targets 50%':               '#800026',
    'Fortify: Oil (Vit A)':                   '#c7e9c0',
    'Fortify: Maize flour (Vit A + Ca)':      '#74c476',
    'Fortify: Full compliance (oil + maize)': '#238b45',
    'Fortify: Full + Wheat flour (Ca)':       '#00441b',
}

for ax, nut in zip(axes, TARGET_NUTRIENTS):
    vals   = all_summary[nut]
    colors = [POLICY_COLORS.get(k, '#cccccc') for k in vals.index]
    bars   = ax.barh(vals.index, vals.values, color=colors,
                     edgecolor='white', linewidth=0.5)
    ax.axvline(vals['Baseline'], color='black', linewidth=1.2,
               linestyle='--', alpha=0.6)
    ax.set_title(nut, fontsize=11, fontweight='bold')
    ax.set_xlabel('% of households meeting RDI')
    ax.set_xlim(0, 100)
    for bar, val in zip(bars, vals.values):
        ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}%', va='center', fontsize=7.5)

# Hide the unused 6th panel if TARGET_NUTRIENTS has fewer than 6 entries
for i in range(len(TARGET_NUTRIENTS), len(axes)):
    axes[i].set_visible(False)

legend_patches = [
    mpatches.Patch(color='#888888', label='Baseline'),
    mpatches.Patch(color='#6baed6', label='Income transfer (blue shades)'),
    mpatches.Patch(color='#e31a1c', label='Price subsidy (red/orange shades)'),
    mpatches.Patch(color='#238b45', label='Fortification (green shades)'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=4, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle(
    'Policy Simulation Results — % of Households Meeting Nutrient RDI\nTanzania Household Sample',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('tanzania_policy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Improvement over baseline (percentage point gains) ──────────────
fig2, axes2 = plt.subplots(2, 3, figsize=(20, 12))
axes2 = axes2.flatten()

for ax, nut in zip(axes2, TARGET_NUTRIENTS):
    gains  = all_summary[nut] - all_summary.loc['Baseline', nut]
    gains  = gains.drop('Baseline')
    colors = ['#d73027' if g < 0 else '#1a9850' for g in gains]
    bars   = ax.barh(gains.index, gains.values, color=colors,
                     edgecolor='white', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_title(nut, fontsize=11, fontweight='bold')
    ax.set_xlabel('Percentage point change vs. baseline')
    for bar, val in zip(bars, gains.values):
        xpos = val + 0.2 if val >= 0 else val - 0.2
        ha   = 'left'    if val >= 0 else 'right'
        ax.text(xpos, bar.get_y() + bar.get_height() / 2,
                f'{val:+.1f}pp', va='center', ha=ha, fontsize=7.5)

for i in range(len(TARGET_NUTRIENTS), len(axes2)):
    axes2[i].set_visible(False)

fig2.suptitle(
    'Improvement Over Baseline (percentage points)\nTanzania Policy Simulations',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('tanzania_policy_gains.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: Heatmap of percentage point changes over ACTUAL baseline ────────

# Actual expenditure baseline adequacy rates (from Chunk 9 output)
# These are the ground-truth adequacy rates from observed expenditures
actual_baseline = pd.Series({
    'Vitamin A, RAE':  12.0,
    'Calcium, Ca':      2.2,
    'Iron, Fe':        49.5,
    'Zinc, Zn':        46.7,
    'Vitamin B-12':    17.2,
    'Potassium, K':    13.0,
})

# Compute percentage point changes from predicted baseline for each scenario
# Then anchor those changes to the actual baseline
predicted_baseline_pct = all_summary.loc['Baseline', TARGET_NUTRIENTS]
pp_changes = all_summary[TARGET_NUTRIENTS].subtract(predicted_baseline_pct, axis=1)

# Drop the baseline row — its change is zero by definition
pp_changes = pp_changes.drop('Baseline')

# Anchored rates: actual baseline + model-estimated change
anchored = pp_changes.add(actual_baseline[TARGET_NUTRIENTS], axis=1).clip(upper=100)

fig3, ax3 = plt.subplots(figsize=(13, 9))

vmin_val = pp_changes.values.min()
vmax_val = pp_changes.values.max()
if vmin_val >= 0:
    vmin_val = -0.1
if vmax_val <= 0:
    vmax_val = 0.1

im = ax3.imshow(pp_changes.values, aspect='auto', cmap='RdYlGn',
                norm=mcolors.TwoSlopeNorm(vmin=vmin_val,
                                          vcenter=0,
                                          vmax=vmax_val))

ax3.set_xticks(range(len(TARGET_NUTRIENTS)))
ax3.set_xticklabels(TARGET_NUTRIENTS, rotation=30, ha='right', fontsize=9)
ax3.set_yticks(range(len(pp_changes)))
ax3.set_yticklabels(pp_changes.index, fontsize=9)

# Annotate each cell with: change value and anchored absolute rate
for i in range(len(pp_changes)):
    for j, nut in enumerate(TARGET_NUTRIENTS):
        delta    = pp_changes.iloc[i, j]
        anchored_val = anchored.iloc[i, j]
        # Top line: the pp change; bottom line: what that means in real terms
        cell_text = f'{delta:+.1f}pp\n({anchored_val:.1f}%)'
        brightness = abs(delta) / (pp_changes.values.max() + 1e-6)
        txt_color  = 'white' if brightness > 0.5 else 'black'
        ax3.text(j, i, cell_text, ha='center', va='center',
                 fontsize=7.5, color=txt_color, linespacing=1.4)

# Actual baseline as a reference row along the bottom
for j, nut in enumerate(TARGET_NUTRIENTS):
    ax3.text(j, len(pp_changes) - 0.3, f'Actual\nbaseline:\n{actual_baseline[nut]:.1f}%',
             ha='center', va='top', fontsize=7, color='#333333',
             style='italic', transform=ax3.transData)

ax3.set_ylim(len(pp_changes) + 1.2, -0.5)  # extend bottom to make room

plt.colorbar(im, ax=ax3, label='Percentage point change vs. predicted baseline')
ax3.set_title(
    'Policy Simulation — Percentage Point Changes in Households Meeting RDI\n'
    'Cell format: [model change] (anchored real-world estimate)',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig('tanzania_policy_heatmap_changes.png', dpi=150, bbox_inches='tight')
plt.show()